# LeRobot Dataset Inspector
Loads the exported dataset through the LeRobot framework, then visualises episodes.

**Install once:**
```bash
pip install lerobot
```

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

DATASET_ROOT = pathlib.Path("./lerobot_dataset")

## Load dataset

In [ ]:
dataset = LeRobotDataset(
    repo_id=str(DATASET_ROOT),   # local path used as repo_id for local-only load
    root=DATASET_ROOT,
    local_files_only=True,
)
print(dataset)

## Dataset summary

In [ ]:
print(f"FPS            : {dataset.fps}")
print(f"Episodes       : {dataset.num_episodes}")
print(f"Total frames   : {dataset.num_frames}")
print(f"Features       : {list(dataset.features.keys())}")
print()
print("Tasks:")
for t in dataset.meta.tasks.values():
    print(f"  {t}")

## Inspect one episode

In [ ]:
EPISODE_IDX = 0   # change to inspect a different episode

ep_dataset = dataset.episode(EPISODE_IDX)
print(f"Episode {EPISODE_IDX}: {len(ep_dataset)} frames")

In [ ]:
# Pull all frames into arrays
obs_state = np.stack([ep_dataset[i]["observation.state"].numpy() for i in range(len(ep_dataset))])
action     = np.stack([ep_dataset[i]["action"].numpy()           for i in range(len(ep_dataset))])
timestamps = np.array([ep_dataset[i]["timestamp"].item()         for i in range(len(ep_dataset))])

print(f"obs_state shape : {obs_state.shape}")   # (N, 6)
print(f"action shape    : {action.shape}")       # (N, 7)
print(f"time range      : 0 → {timestamps[-1]:.2f}s")

## Joint positions: commanded vs actual

In [ ]:
cmd    = action[:, :6]
actual = obs_state

fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
for j, ax in enumerate(axes.flatten()):
    ax.plot(timestamps, cmd[:, j],    label="cmd",    lw=1.2)
    ax.plot(timestamps, actual[:, j], label="actual", lw=1.2, linestyle="--")
    ax.set_title(f"J{j+1}")
    ax.set_ylabel("deg")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1, 0].set_xlabel("time (s)")
axes[-1, 1].set_xlabel("time (s)")
fig.suptitle(f"Episode {EPISODE_IDX} — Joint positions", fontsize=12)
plt.tight_layout()
plt.show()

## Gripper

In [ ]:
gripper = action[:, 6]

plt.figure(figsize=(10, 3))
plt.plot(timestamps, gripper, lw=1.5)
plt.axhline(0.65, color="green",  ls="--", lw=0.9, label="open threshold")
plt.axhline(0.35, color="orange", ls="--", lw=0.9, label="close threshold")
plt.ylim(-0.05, 1.05)
plt.xlabel("time (s)"); plt.ylabel("gripper_norm")
plt.title(f"Episode {EPISODE_IDX} — Gripper")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## EEF pose

In [ ]:
eef = np.stack([ep_dataset[i]["observation.eef_pose"].numpy() for i in range(len(ep_dataset))])
labels = ["x_mm", "y_mm", "z_mm", "rx_deg", "ry_deg", "rz_deg"]

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for i, (ax, lbl) in enumerate(zip(axes.flatten(), labels)):
    ax.plot(timestamps, eef[:, i], lw=1.2)
    ax.set_title(lbl); ax.grid(True, alpha=0.3)
    if i >= 3: ax.set_xlabel("time (s)")
fig.suptitle(f"Episode {EPISODE_IDX} — EEF pose", fontsize=12)
plt.tight_layout(); plt.show()

## 3-D EEF path

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection="3d")
sc  = ax.scatter(eef[:, 0], eef[:, 1], eef[:, 2], c=timestamps, cmap="viridis", s=4)
plt.colorbar(sc, ax=ax, label="time (s)")
ax.set_xlabel("X (mm)"); ax.set_ylabel("Y (mm)"); ax.set_zlabel("Z (mm)")
ax.set_title(f"Episode {EPISODE_IDX} — EEF 3-D path")
plt.tight_layout(); plt.show()

## Show wrist camera frame

In [ ]:
frame_idx = 0   # change to any frame index

sample = ep_dataset[frame_idx]
if "observation.images.wrist_cam" in sample:
    img = sample["observation.images.wrist_cam"]   # (C, H, W) float tensor [0,1] or uint8
    img_np = img.numpy()
    if img_np.ndim == 3 and img_np.shape[0] in (1, 3):
        img_np = img_np.transpose(1, 2, 0)          # CHW → HWC
    if img_np.dtype != np.uint8:
        img_np = (img_np * 255).clip(0, 255).astype(np.uint8)
    plt.figure(figsize=(6, 4))
    plt.imshow(img_np)
    plt.axis("off")
    plt.title(f"Episode {EPISODE_IDX}, frame {frame_idx}")
    plt.tight_layout(); plt.show()
else:
    print("No camera frames in this episode (camera was not attached during recording).")

## Dataset-wide statistics

In [ ]:
durations, frame_counts, task_labels = [], [], []
task_names = {t["task_index"]: t["task"] for t in dataset.meta.info["tasks"]}

for ep_idx in range(dataset.num_episodes):
    ep = dataset.episode(ep_idx)
    ts = np.array([ep[i]["timestamp"].item() for i in range(len(ep))])
    durations.append(float(ts[-1]))
    frame_counts.append(len(ep))
    tidx = int(ep[0]["task_index"].item())
    task_labels.append(task_names.get(tidx, str(tidx)))

task_counts = {lbl: task_labels.count(lbl) for lbl in set(task_labels)}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(durations,    bins=20, color="steelblue",   edgecolor="white")
axes[0].set_xlabel("duration (s)"); axes[0].set_title("Episode durations")
axes[1].hist(frame_counts, bins=20, color="salmon",      edgecolor="white")
axes[1].set_xlabel("frames");       axes[1].set_title("Frames per episode")
short = [l[:28]+"..." if len(l)>28 else l for l in task_counts]
axes[2].bar(range(len(task_counts)), list(task_counts.values()), color="mediumpurple", edgecolor="white")
axes[2].set_xticks(range(len(task_counts)))
axes[2].set_xticklabels(short, rotation=20, ha="right", fontsize=8)
axes[2].set_title("Episodes per task")
plt.tight_layout(); plt.show()

print(f"Mean duration : {np.mean(durations):.2f}s")
print(f"Mean frames   : {np.mean(frame_counts):.0f}")